11 de noviembre 2026. (Estoy en semana de final jeje)

Existe un cambio en la Encuesta a partir del 2019 en adelante asi que por ahora limpiamos y tratamos la base de datos de 2017 a 2019 con el fin de extraer unas aproximaciones de calidad antes de la expancion del SIS.

Importar datos de inei
https://github.com/TJhon/PyPeruStats

leer archivos .SAV
https://pypi.org/project/pyreadstat/

In [23]:

#librerias para descargar el endes de inei
from perustats.inei import INEIFetcher #pip install perustats

#Librerias para tratar datos
import pyreadstat as pdst 
import numpy as np
import pandas as pd
from functools import reduce

#Librerias visualizacion
import matplotlib.pyplot as plt 
import seaborn as sns

#Librerias estimacion
import statsmodels.api as sm
import sklearn as sk
from scipy.optimize import minimize
from scipy.stats import chi2, norm

#Pa saber dondtoy y si mi ruta existe o no :v:V:VV:V::V:
from pathlib import Path
import os as os

In [24]:
(INEIFetcher("endes", years=range(2017, 2020))
    .fetch_modules()
    .download(module_codes=[64, 65,66,69,70,73, 414,569])
    .organize(organize_by="module", operation="move"))

Output()

✓ Found 39 modules for ENDES

Output()

Output()

Output()

Files organized under data\microodatos_inei\endes\2_organized\by_module

INEIFetcher(survey='endes', years=[2017, 2018, 2019], modules=39)

In [25]:

archivos=(["0064_caracteristicas_del_hogar",
"0065_caracteristicas_de_la_vivienda",
"0066_datos_basicos_de_mef",
"0069_embarazo_parto_puerperio_y_lactancia",
"0070_inmunizacin_y_salud",
"0073_mortalidad_materna_-_violencia_familiar",
"0414_encuesta_de_salud"
])
en1=["h0","h1","h4","h23","41","42","43","83","84dv","91","94","95","0111","hm"] 
en2=["csalud01","csalud03","csalud07","csalud08","dit"]
encuesta= ["rec"+i for i in en1] +en2

In [26]:
dir0 = Path.cwd()
nombres=[]
df=[]
n_datas=0
for i in [2017,2018,2019]:
    for j in archivos:
        for k in encuesta:
            ruta=dir0 / r"data\microodatos_inei\endes\2_organized\by_module"/ j / f"{i}_{k}.sav"
            if os.path.exists(ruta):
                n_datas+=1
                nombres.append(f"{i}_{k}")
                df.append(pdst.read_sav(ruta, encoding="latin1")[0]) #Lo de , encoding="latin1" salio de github xdxdxd https://github.com/tidyverse/haven/issues/615
            else:
                continue
print("correcto" if n_datas==len(df) else "Tienes que revisar.")

correcto


In [39]:
pd.DataFrame(nombres).transpose()


,0,1,2,3,4,5,6,7,8,9,...,39,40,41,42,43,44,45,46,47,48
0,2017_rech0,2017_rech1,2017_rech4,2017_rech23,2017_rec91,2017_rec0111,2017_rec41,2017_rec94,2017_rec42,2017_rec43,...,2019_rec41,2019_rec94,2019_rec42,2019_rec43,2019_rec95,2019_dit,2019_rec83,2019_rec84dv,2019_csalud01,2019_csalud08


In [ ]:




año=0

for i, j in enumerate(nombres):
    encuesta=j[5:]
    print(j,encuesta,año)
    if año==int(j[3]):
        
    año=int(j[3])
    encuesta=j[5:]
    


2017_rech0 rech0 7
2017_rech1 rech1 7
2017_rech4 rech4 7
2017_rech23 rech23 7
2017_rec91 rec91 7
2017_rec0111 rec0111 7
2017_rec41 rec41 7
2017_rec94 rec94 7
2017_rec42 rec42 7
2017_rec43 rec43 7
2017_rec95 rec95 7
2017_rec83 rec83 7
2017_rec84dv rec84dv 7
2017_csalud01 csalud01 7
2017_csalud08 csalud08 7
2018_rech0 rech0 8
2018_rech1 rech1 8
2018_rech4 rech4 8
2018_rechm rechm 8
2018_rech23 rech23 8
2018_rec91 rec91 8
2018_rec0111 rec0111 8
2018_rec41 rec41 8
2018_rec94 rec94 8
2018_rec42 rec42 8
2018_rec43 rec43 8
2018_rec95 rec95 8
2018_dit dit 8
2018_rec83 rec83 8
2018_rec84dv rec84dv 8
2018_csalud01 csalud01 8
2018_csalud08 csalud08 8
2019_rech0 rech0 9
2019_rech1 rech1 9
2019_rech4 rech4 9
2019_rechm rechm 9
2019_rech23 rech23 9
2019_rec91 rec91 9
2019_rec0111 rec0111 9
2019_rec41 rec41 9
2019_rec94 rec94 9
2019_rec42 rec42 9
2019_rec43 rec43 9
2019_rec95 rec95 9
2019_dit dit 9
2019_rec83 rec83 9
2019_rec84dv rec84dv 9
2019_csalud01 csalud01 9
2019_csalud08 csalud08 9


In [ ]:
cod_hogar=[['rech0',['HHID']],
           ['rech23',['HHID']]
           ]

cod_miembro=[[['rech1',["HHID","HVIDX"]],
             ['rech4',["HHID","IDXH4"]]
             ],["HHID"]]

cod_mujer=[[['rec91',['CASEID']],
           ['rec0111',['CASEID']],
           ['rec42',['CASEID']],
           ['rec84dv',['CASEID']],
           ],['HHID']]

cod_niños=[[['rec41',["CASEID","MIDX"]],
           ['rec94',["CASEID","IDX94"]],
           ['rec43',["CASEID","HIDX"]],
           ['rec95',["CASEID","IDX95"]],
           ],['CASEID']]
cod_s_adulto=[['csalud01'],[["QHCLUSTER","QHNUMBER","QHHOME"],["HV001","HV002","HV002A"]]]
cod_s_niño=[['csalud08'],[["QHCLUSTER","QHNUMBER","QHHOME"],["HV001","HV002","HV002A"]]]

años=['2017','2018','2019']
